# 📖 EDA & Preprocessing

## Описание проекта

Этот ноутбук является частью пет-проекта по fine-tuning языковой модели 
**EleutherAI Pythia** на корпусе народных сказок, историй и сказок братьев Гримм.

Цель модели — генерировать связный текст сказки по заданному названию.

---

## Что делается в этом ноутбуке

1. **Загрузка и анализ датасетов** - статистика по длинам текстов в словах и токенах
2. **Обработка txt файлов** - извлечение заголовков, обнаружение фрагментов
3. **Оценка полноты сюжета** - через локальную LLM (Qwen 7B Instruct via LM Studio)
4. **Генерация названий** - для сказок без заголовка
5. **Сборка финального датасета** - склейка источников в единый JSONL файл

---


## Источники данных

| Датасет | Источник | Формат |
|---|---|---|
| Fairy tales from around the world | https://www.kaggle.com/datasets/annbengardt/fairy-tales-from-around-the-world | TXT  |
| 1000+ Folk Stories around the World | https://www.kaggle.com/datasets/chayanonc/1000-folk-stories-around-the-world | CSV |
| Grimms' Fairy Tales | https://www.kaggle.com/datasets/tschomacker/grimms-fairy-tales| CSV |
| Folk Tales | https://www.kaggle.com/datasets/thedevastator/folk-tales| CSV |

---

## Стек

- **Python 3.12**, pandas, numpy
- **HuggingFace Transformers** - токенизатор Pythia
- **LM Studio** (локальный сервер) - Qwen 7B Instruct для оценки и генерации заголовков
- **PyTorch 2.5.1 + CUDA 12.1** - RTX 3090

In [1]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
from config import CHUNKED_JSONL_PATH, BASE_MODEL_NAME
import torch
import os
import json
from openai import OpenAI
from pathlib import Path
from typing import Union
from tqdm import tqdm

In [3]:
print("PyTorch version:", torch.__version__)
print("CUDA version in PyTorch:", torch.version.cuda)
print("Is CUDA available?:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

PyTorch version: 2.5.1+cu121
CUDA version in PyTorch: 12.1
Is CUDA available?: True
Device count: 1


In [5]:
# Используем локальную модель Qwen 7b Instruct для оценки сюжета и генерации названий
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# BASE_MODEL_NAME - модель которую обучаем, берем её токенизатор
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# Обработка датасетов


In [6]:
def df_to_records(df, title_col = "title", story_col ="text"):
    """Конвертирует один DataFrame в список записей.

    - df: исходный DataFrame
    - title_col: название колонки с заголовками сказок (по умолчанию "title")
    - story_col: название колонки с текстами сказок (по умолчанию "text")
    """
    df.columns = df.columns.str.lower()
    records = []
    for _, row in df.iterrows():
        title = str(row[title_col]).strip()
        story = str(row[story_col]).strip()
        if title and story:
            records.append({"title": title, "completion": story})
    return records


def write_jsonl(records, output=None):
    """Записывает список записей в JSONL файл."""
    if output:
        with open(output, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

def read_jsonl(file_path):
    """Читает JSONL файл и возвращает список записей (словарей)."""
    records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip(): # Пропускаем пустые строки, если есть
                records.append(json.loads(line))
    return records

In [7]:
# Генерация названия сказки по содержанию
def generate_title(story_text):
    response = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user", "content": f"Give a short title (3-6 words) for this fairy tale:\n\n{story_text}\n\nTitle:"}],
        max_tokens=50,
        temperature=0.8,
    )
    return response.choices[0].message.content.strip()

# оценить целостность сюжета
def evaluate_story(story_text):
    response = client.chat.completions.create(
        model="local-model",
        messages=[
            {"role": "user", "content": f"""Evaluate this fairy tale text and answer these questions briefly:
1. Is this a complete story (has beginning, middle, end)?
2. Is the plot coherent and logical?
3. Does it look like a fragment of a larger text?

Answer with: COMPLETE, FRAGMENT, or UNCLEAR. Only one word.

Text:
{story_text}"""}
        ],
        max_tokens=50,
        temperature=0.1,
    )
    return response.choices[0].message.content.strip()


In [8]:
def parse_and_detect_titles(input_dir):
    """Быстрый этап: чтение файлов и поиск существующих заголовков."""
    dataset = []
    folder = Path(input_dir)
    
    print("Этап 1: Чтение файлов и поиск заголовков...")
    # list() нужен, чтобы tqdm знал общее количество файлов
    for file_path in tqdm(list(folder.glob("*.txt"))):
        try:
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read().lstrip()
            except UnicodeDecodeError:
                with open(file_path, "r", encoding="windows-1251") as f:
                    content = f.read().lstrip()
                    
            if not content:
                continue
                
            parts = content.split("\n\n", 1)
            
            # Проверяем наличие заголовка
            if len(parts) == 2 and parts[0].isupper():
                dataset.append({
                    "file_name": file_path.name,
                    "title": parts[0].strip(),
                    "completion": parts[1].strip(),
                    "needs_title": False # Заголовок уже есть
                })
            else:
                dataset.append({
                    "file_name": file_path.name,
                    "title": None,
                    "completion": content.strip(),
                    "needs_title": True # Заголовок нужно сгенерировать
                })
                
        except Exception as e:
            print(f"Ошибка чтения {file_path.name}: {e}")
            
    return dataset
def evaluate_dataset_plot(dataset):
    """Медленный этап: оценка связности сюжета через локальную LLM."""
    print("Этап 2: Оценка полноты сюжета...")
    
    evaluated_dataset_new_1 = []
    
    for item in tqdm(dataset):
        # Отправляем текст в твою функцию evaluate_story
        status = evaluate_story(item["completion"]).upper()
        
        item["status"] = status
        evaluated_dataset_new_1.append(item)
        
    return evaluated_dataset_new_1

def generate_missing_titles(dataset):
    """Медленный этап: генерация заголовков только для безымянных и полных сказок."""
    print("Этап 3: Генерация недостающих заголовков...")
    
    final_dataset = []
    
    for item in tqdm(dataset):
        # Если сюжет неполный, нам незачем тратить время на генерацию названия
        if item["status"] != "COMPLETE":
            final_dataset.append(item)
            continue
            
        if item["needs_title"]:
            generated_title = generate_title(item["completion"])
            item["title"] = generated_title
            
        final_dataset.append(item)
        
    return final_dataset

## Dataset fairy-tales

Источник https://www.kaggle.com/datasets/annbengardt/fairy-tales-from-around-the-world

- Множество файлов .txt, в некоторых сказка целиком и с названием (первые 175)

- В некоторых без названия

- Также есть сказки которые разбиты на несколько файлов, склейка производилась вручную, целостность  сюжета оценивалась той же моделью

- По итогу склеивает всё в один файл fairy_tales.jsonl

In [9]:
def parse_and_detect_titles(input_dir):
    """Быстрый этап: чтение файлов и поиск существующих заголовков."""
    dataset = []
    folder = Path(input_dir)
    
    print("Этап 1: Чтение файлов и поиск заголовков...")
    # list() нужен, чтобы tqdm знал общее количество файлов
    for file_path in tqdm(list(folder.glob("*.txt"))):
        try:
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read().lstrip()
            except UnicodeDecodeError:
                with open(file_path, "r", encoding="windows-1251") as f:
                    content = f.read().lstrip()
                    
            if not content:
                continue
                
            parts = content.split("\n\n", 1)
            
            # Проверяем наличие заголовка
            if len(parts) == 2 and parts[0].isupper():
                dataset.append({
                    "file_name": file_path.name,
                    "title": parts[0].strip(),
                    "completion": parts[1].strip(),
                    "needs_title": False # Заголовок уже есть
                })
            else:
                dataset.append({
                    "file_name": file_path.name,
                    "title": None,
                    "completion": content.strip(),
                    "needs_title": True # Заголовок нужно сгенерировать
                })
                
        except Exception as e:
            print(f"Ошибка чтения {file_path.name}: {e}")
            
    return dataset


In [10]:
raw_dataset = parse_and_detect_titles("./dataset/fairy_tales")
print(f"Найдено текстов: {len(raw_dataset)}")

Этап 1: Чтение файлов и поиск заголовков...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1650/1650 [00:00<00:00, 37125.70it/s]

Найдено текстов: 1650


In [11]:
evaluated_dataset = evaluate_dataset_plot(raw_dataset)

clean_tales = [item for item in evaluated_dataset if item.get("status") == "COMPLETE"]
write_jsonl(clean_tales, "./dataset/fairy_tales/complete_tales.jsonl")

bad_tales = [item for item in evaluated_dataset if item.get("status") != "COMPLETE"]

# Сортируем список
# lambda извлекает число из имени файла: "375.txt" -> "375" -> 375
bad_tales_sorted = sorted(
    bad_tales, 
    key=lambda item: int(item["file_name"].split(".")[0])
)

write_jsonl(bad_tales_sorted, "./dataset/fairy_tales/bad_tales_sorted.jsonl")


print(f"Готово! Сохранено {len(clean_tales)} идеальных сказок.")
print(f"Готово! Сохранено {len(bad_tales_sorted)} битых сказок.")

Этап 2: Оценка полноты сюжета...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1650/1650 [22:30<00:00,  1.22it/s]

Готово! Сохранено 1419 идеальных сказок.
Готово! Сохранено 231 битых сказок.


In [12]:

final_clean_tales = generate_missing_titles(clean_tales)
write_jsonl(final_clean_tales, "./dataset/fairy_tales/fairy_tales.jsonl")


Этап 3: Генерация недостающих заголовков...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1419/1419 [18:06<00:00,  1.31it/s]


## Обработка folk_tales.csv

In [13]:
df_folk_tales = pd.read_csv("dataset/folk_tales.csv")  
folk_tales_records = df_to_records(df_folk_tales)
write_jsonl(folk_tales_records, "dataset/folk_tales.jsonl")

## Обработка датасета 1000Folk_Story_around_the_Globe.csv


In [14]:
df_1000_folk = pd.read_csv("dataset/1000Folk_Story_around_the_Globe.csv")
folk_1000 = df_to_records(df_1000_folk,title_col = "title", story_col ="full_text")
write_jsonl(folk_1000, "dataset/folk_1000.jsonl")

## Обработка датасет grimms_fairytales.csv

In [15]:
df_grimms = pd.read_csv("dataset/grimms_fairytales.csv")
folk_grimms = df_to_records(df_grimms,title_col = "title", story_col ="text")
write_jsonl(folk_grimms, "dataset/folk_grimms.jsonl")

In [16]:
import json

def merge_and_validate_jsonl(input_files: list, output_file: str, tokenizer):
    """
    Склеивает JSONL файлы, валидирует, удаляет дубликаты и добавляет token_count.
    """
    seen_titles = set()
    valid_count = 0
    error_count = 0
    duplicate_count = 0
    
    with open(output_file, 'w', encoding='utf-8') as out_f:
        for file_path in input_files:
            print(f"Обработка {file_path}...")
            
            with open(file_path, 'r', encoding='utf-8') as in_f:
                for line_num, line in enumerate(in_f, 1):
                    line = line.strip()
                    if not line:
                        continue
                        
                    try:
                        record = json.loads(line)
                        
                        title = str(record.get("title", "")).strip()
                        completion = str(record.get("completion", "")).strip()
                        
                        if not title or not completion:
                            error_count += 1
                            continue 
                            
                        title_lower = title.lower()
                        if title_lower in seen_titles:
                            duplicate_count += 1
                            continue
                        seen_titles.add(title_lower)
                        
                        # --- НОВАЯ ЛОГИКА ---
                        # Считаем токены именно токенизатором модели
                        # add_special_tokens=False, так как спец-токены добавятся при обучении
                        token_count = len(tokenizer(completion, add_special_tokens=False)["input_ids"])
                        
                        clean_record = {
                            "title": title, 
                            "completion": completion,
                            "token_count": token_count # Сохраняем метаданные
                        }
                        # --------------------
                        
                        out_f.write(json.dumps(clean_record, ensure_ascii=False) + '\n')
                        valid_count += 1
                        
                    except json.JSONDecodeError:
                        error_count += 1
                        print(f"Ошибка парсинга JSON: {file_path}, строка {line_num}")
                        
    print("-" * 30)
    print("Склейка завершена!")
    print(f"Идеальных сказок сохранено: {valid_count}")
    print(f"Удалено дубликатов: {duplicate_count}")
    print(f"Отброшено битых строк: {error_count}")


files_to_merge = [
    "dataset/fairy_tales/fairy_tales.jsonl",
    "dataset/folk_tales.jsonl",
    "dataset/folk_1000.jsonl",
    "dataset/folk_grimms.jsonl"
]

# Запускаем сборку
merge_and_validate_jsonl(files_to_merge, "dataset/master_dataset_clean.jsonl", tokenizer)

Обработка dataset/fairy_tales/fairy_tales.jsonl...
Обработка dataset/folk_tales.jsonl...
Обработка dataset/folk_1000.jsonl...
Обработка dataset/folk_grimms.jsonl...
------------------------------
Склейка завершена!
Идеальных сказок сохранено: 4355
Удалено дубликатов: 1371
Отброшено битых строк: 0


###  Some helpful functions

In [18]:
#  Статистика количества токенов по набору данных
def stat_info_jsonl(filepath_jsonl):
    char_lengths = []
    token_lengths = []
    # result_dataset.jsonl
    with open(filepath_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            completion = record.get("completion", "")
            char_lengths.append(len(completion))
            token_lengths.append(len(tokenizer(completion, add_special_tokens=False)["input_ids"]))
    
    char_lengths = np.array(char_lengths)
    token_lengths = np.array(token_lengths)
    
    print(f"Всего сказок: {len(char_lengths)}\n")
    
    print("=== Символы ===")
    print(f"Мин:     {char_lengths.min()}")
    print(f"Макс:    {char_lengths.max()}")
    print(f"Среднее: {char_lengths.mean():.0f}")
    print(f"Медиана: {np.median(char_lengths):.0f}")
    
    print("\n=== Токены ===")
    print(f"Мин:     {token_lengths.min()}")
    print(f"Макс:    {token_lengths.max()}")
    print(f"Среднее: {token_lengths.mean():.0f}")
    print(f"Медиана: {np.median(token_lengths):.0f}")
    print(f"95%:     {np.percentile(token_lengths, 95):.0f}")
    
    print("\n=== Влезают в контекст ===")
    for limit in [512, 1024, 1536, 2048, 4096, 8192]:
        count = np.sum(token_lengths <= limit)
        pct = count / len(token_lengths) * 100
        print(f"  <= {limit} токенов: {count}/{len(token_lengths)} ({pct:.1f}%)")



In [19]:
stat_info_jsonl("dataset/master_dataset_clean.jsonl")

Всего сказок: 4355

=== Символы ===
Мин:     130
Макс:    106121
Среднее: 9715
Медиана: 7123

=== Токены ===
Мин:     39
Макс:    25091
Среднее: 2350
Медиана: 1735
95%:     6462

=== Влезают в контекст ===
  <= 512 токенов: 706/4355 (16.2%)
  <= 1024 токенов: 1297/4355 (29.8%)
  <= 1536 токенов: 1936/4355 (44.5%)
  <= 2048 токенов: 2467/4355 (56.6%)
  <= 4096 токенов: 3681/4355 (84.5%)
  <= 8192 токенов: 4251/4355 (97.6%)
